# AMD-Latent
### Latent space exploration

---

The current notebook applies different approaches to explore RETCLIP latent space.

# Packages import and functions defining

Loading the required packages and definition of functions.

In [ ]:
# PACKAGES IMPORT

# System packages
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Jupyter packages
from ipywidgets import widgets
from amdlatent.utils.widgets import toggleoptions, filechooser, slideroptions

# Dataset packages
from amdlatent.dataset.audit import naneval, correlationanalysis, bhfdr, patientbootstrapindex, bootstrapci
from amdlatent.dataset.extraction import filteringimages, findsmokingstatus

# Plotting packages
import matplotlib.pyplot as plt

# Analysis packages
import seaborn as sns
from amdlatent.utils.visualization import amdscorelabeler, drusensizecategorizer, drusenareacategorizer, increasedpigmentcategorizer, depigmentationcategorizer, geographiccentercategorizer, geographicareacategorizer, geographicgridcategorizer, hemorrhagiccategorizer, exudatecategorizer, subhemorragecategorizer, fibrosiscategorizer, glaucomalabeler, diabeteslabeler, agelabeler, sexlabeler, bmilabeler, classcentroids, interclassdistances, intraclassvariability, discriminabilityratio, distancesbootstrap, extendedseparation

# Statistical analysis packages
from sklearn.preprocessing import normalize
from scipy.stats import spearmanr

# Path checker
if Path().resolve().name != "AMD-Latent": os.chdir(os.path.join(Path().resolve().parents[1]))

# Code execution

Select the following options for running the script and execute it.

In [ ]:
# OPTIONS SELECTION

dirselect = filechooser("Select the <b>AMD Study</b> folder:")
clipselect = toggleoptions(['RETCLIP', 'CLIP', 'ResNet50'], "Select the <b>model</b> to use:")

modeselect = toggleoptions(['Distances', 'Correlation'], "Select the <b>mode</b> to run:")

clusterbox = widgets.Output()
display(clusterbox)
infoselect = None
statselect = None
bootselect = None
extendedselect = None

eval_value = 'Extraction'
info_value = None

def _on_info_change(change):
    """Smart observer: Only re-renders if slider state actually needs to change."""
    global info_value
    old_val = change['old']
    new_val = change['new']
    info_value = new_val

def _update_clusterselect(change=None):
    global infoselect, statselect, percentselect, eval_value, info_value, bootselect, extendedselect
        
    clusterbox.clear_output(wait=True)
    percentselect = infoselect = statselect = None
    bootselect = extendedselect = None
    mode = modeselect.value

    with clusterbox:
        show_info = False
        if mode in ["Correlation", "Distances"]:
            show_info = True
            
        if show_info:
            base_info_options = [
                'AMD', 'Age', 'Sex', 'BMI', 'Smoking', 'Drusen Area', 'Drusen Size', 
                'Increased Pigmentation', 'Depigmentation', 'Geographic Center', 
                'Geographic Area', 'Geographic Grid', 'Hemorrhagic', 'Exudate', 
                'Subhemorrage', 'Fibrosis', 'Glaucoma', 'Diabetes'
            ]
            
            info_options = base_info_options

            infoselect = toggleoptions(
                info_options, 
                "Select the <b>type of information</b> to evaluate in the clusters:"
            )
            if info_value in info_options: infoselect.value = info_value
            else: info_value = infoselect.value
            infoselect.observe(_on_info_change, names="value")
            if mode == "Correlation":
                statselect = toggleoptions(
                    ['Spearman', 'Mann-Whitney', 'Brunner-Munzel'], 
                    "Select the <b>statistic method</b> to use for correlation evaluation:"
                )
                bootselect = slideroptions(
                    "Select the <b>number of patient-cluster bootstrap replicates</b> for the BH-FDR-significant associations (0 = skip):",
                    0, 2000, 50, 100
                )
            elif mode == "Distances":
                bootselect = slideroptions(
                    "Select the <b>number of patient-cluster bootstrap replicates</b> for 95% CIs (0 = point estimates only):",
                    0, 2000, 50, 200
                )
                extendedselect = toggleoptions(
                    ['No', 'Yes'],
                    "Also compute <b>extended separation metrics</b> (Mahalanobis / Frechet / MMD, slower)?:"
                )

def _bind_clusterselect():
    if modeselect is not None:
        modeselect.observe(_update_clusterselect, names="value")
    _update_clusterselect()

_bind_clusterselect()

In [ ]:
# SCRIPT EXECUTION

metadata = {
    "directory": dirselect.selected_path,
    "cliptype": clipselect.value,
    "mode": modeselect.value,
    "info": infoselect.value if infoselect is not None else None,
    "stat": statselect.value if statselect is not None else None,
    "boot": bootselect.value if bootselect is not None else None,
    "extended": extendedselect.value if extendedselect is not None else None
}

dataset = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='AMD')
dataset.columns = dataset.columns.astype(str).str.strip()
fundus = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='fundus')
fundus.columns = fundus.columns.astype(str).str.strip()
followup = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='followup')
followup.columns = followup.columns.astype(str).str.strip()
enrollment = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='enrollment_randomization')
enrollment.columns = enrollment.columns.astype(str).str.strip()

latents = np.load(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower(), "latentvec.npy"))
paths = np.load(os.path.join("results", "areds", "dataset", "encoding", metadata["cliptype"].lower(), "imagepaths.npy"))
paths = pd.DataFrame(paths)
paths = paths.apply(lambda col: col.map(lambda p: os.path.join(metadata["directory"], "Data", "EDO", *Path(p).parts[-5:])))
blacklist = naneval(os.path.join(metadata["directory"], "Data", "EDO"), dataset)
filtpaths = filteringimages(paths, blacklist)
latents = latents[[i for i, path in enumerate(paths[paths.columns[0]].tolist()) if path in filtpaths[filtpaths.columns[0]].tolist()]]
pathlist = filtpaths[filtpaths.columns[0]].tolist()

if metadata["info"] == 'AMD':
    labels, cmap = amdscorelabeler(pathlist, dataset, 'All')
elif metadata["info"] == 'Drusen Size':
    drusensizeL = [drusensizecategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    drusensizeR = [drusensizecategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(drusensizeL, drusensizeR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(drusensizeL, drusensizeR)]
    cmap = {f"Drusen Size {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Drusen Area':
    drusenareaL = [drusenareacategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    drusenareaR = [drusenareacategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(drusenareaL, drusenareaR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(drusenareaL, drusenareaR)]
    cmap = {f"Drusen Area {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Increased Pigmentation':
    increasedpigmentL = [increasedpigmentcategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    increasedpigmentR = [increasedpigmentcategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(increasedpigmentL, increasedpigmentR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(increasedpigmentL, increasedpigmentR)]
    cmap = {f"Increased Pigmentation {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Depigmentation':
    depigmentationL = [depigmentationcategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    depigmentationR = [depigmentationcategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(depigmentationL, depigmentationR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(depigmentationL, depigmentationR)]
    cmap = {f"Depigmentation {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Geographic Center':
    geographiccenterL = [geographiccentercategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    geographiccenterR = [geographiccentercategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(geographiccenterL, geographiccenterR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(geographiccenterL, geographiccenterR)]
    cmap = {f"Center {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Geographic Area':
    geographicareaL = [geographicareacategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    geographicareaR = [geographicareacategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(geographicareaL, geographicareaR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(geographicareaL, geographicareaR)]
    cmap = {f"Area {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Geographic Grid':
    geographicgridL = [geographicgridcategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    geographicgridR = [geographicgridcategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(geographicgridL, geographicgridR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(geographicgridL, geographicgridR)]
    cmap = {f"Grid {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Hemorrhagic':
    hemorrhagicL = [hemorrhagiccategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    hemorrhagicR = [hemorrhagiccategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(hemorrhagicL, hemorrhagicR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(hemorrhagicL, hemorrhagicR)]
    cmap = {f"Hemorrhagic {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Exudate':
    exudateL = [exudatecategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    exudateR = [exudatecategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(exudateL, exudateR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(exudateL, exudateR)]
    cmap = {f"Exudate {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Subhemorrage':
    subhemorrageL = [subhemorragecategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    subhemorrageR = [subhemorragecategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(subhemorrageL, subhemorrageR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(subhemorrageL, subhemorrageR)]
    cmap = {f"Subhemorrage {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'Fibrosis':
    fibrosisL = [fibrosiscategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "LE", fundus) for p in pathlist]
    fibrosisR = [fibrosiscategorizer(str(Path(p).parts[-5]), int(str(Path(p).parts[-2]).strip()), "RE", fundus) for p in pathlist]
    if metadata["mode"] == "Correlation": labels = [[l, r] for l, r in zip(fibrosisL, fibrosisR)]
    else: labels = ["".join(sorted([left, right])) for left, right in zip(fibrosisL, fibrosisR)]
    cmap = {f"Fibrosis {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
elif metadata["info"] == 'BMI':
    labels, cmap = bmilabeler(pathlist, enrollment)
elif metadata["info"] == 'Age':
    labels, cmap = agelabeler(pathlist, enrollment)
    labels = np.array([(int(x // 5) * 5) for x in labels]).astype(int)
elif metadata["info"] == 'Sex':
    labels, cmap = sexlabeler(pathlist, enrollment)
elif metadata["info"] == 'Glaucoma':
    labels, cmap = glaucomalabeler(pathlist, followup)
elif metadata["info"] == 'Diabetes':
    labels, cmap = diabeteslabeler(pathlist, followup)
elif metadata["info"] == 'Smoking':
    smokingtable = findsmokingstatus(enrollment, followup)
    smokingtable = pd.DataFrame(smokingtable)
    smokingtable = pd.concat([smokingtable] + [smokingtable.iloc[[-1]]] * 10, ignore_index=True)
    labels = [smokingtable.loc[int(str(Path(path).parts[-2]).strip()), str(Path(path).parts[-5])] for path in pathlist]
    cmap = {f"Smoking {i}": plt.cm.viridis(idx / len(np.unique(labels))) for idx, i in enumerate(np.unique(labels))}
else: 
    labels = None
    cmap = None

if metadata["mode"] != 'Distances': latents = (latents - latents.min()) / (latents.max() - latents.min())

if metadata["mode"] == 'Distances':
    patientids = np.array([str(Path(p).parts[-5]) for p in pathlist])
    nboot = int(metadata["boot"]) if metadata["boot"] else 0
    if nboot > 0:
        result = distancesbootstrap(latents, labels, patientids, variant="var", n_replicates=nboot, randomstate=42)
        classes, centroids = result["classes"], result["centroids"]
        interclass, intraclass, fdr = result["interclass"], result["intraclass"], result["fdr"]
        interclass_ci, intraclass_ci, fdr_ci = result["interclass_ci"], result["intraclass_ci"], result["fdr_ci"]
    else:
        classes, centroids = classcentroids(latents, labels)
        interclass = interclassdistances(centroids)
        intraclass = intraclassvariability(latents, labels, classes, centroids)
        fdr = discriminabilityratio(interclass, intraclass["var"])
        interclass_ci = intraclass_ci = fdr_ci = None

    centroidlabels = [f"AMD {c}" for c in classes] if metadata["info"] == 'AMD' else list(classes)
    pairidx = [(i, j) for i in range(len(classes)) for j in range(i + 1, len(classes))]
    interclasslabels = [f"{metadata['info']} {classes[i]} - {metadata['info']} {classes[j]} interclass distance" for i, j in pairidx]
    if interclass_ci is not None:
        interclassdist = pd.DataFrame(
            {
                "Interclass Distance": [interclass[i, j] for i, j in pairidx],
                "95% CI low": [interclass_ci[i, j, 0] for i, j in pairidx],
                "95% CI high": [interclass_ci[i, j, 1] for i, j in pairidx],
            },
            index=interclasslabels,
        )
    else:
        interclassdist = pd.DataFrame([interclass[i, j] for i, j in pairidx], index=interclasslabels, columns=["Interclass Distance"])

    latentsnorm = normalize(latents)
    intraclassdistances = {f"{metadata['info']} {c}": 1.0 - latentsnorm[np.asarray(labels) == c] @ centroids[idx] for idx, c in enumerate(classes)}
    plt.figure(figsize=(10, 5))
    if intraclass_ci is not None:
        lower = np.clip(intraclass["mean"] - intraclass_ci["mean"][:, 0], 0, None)
        upper = np.clip(intraclass_ci["mean"][:, 1] - intraclass["mean"], 0, None)
        yerr = [lower, upper]
    else:
        yerr = np.sqrt(intraclass["var"])
    plt.bar(x=centroidlabels, height=intraclass["mean"], yerr=yerr, capsize=4, color = [cmap[i] for i in cmap.keys()])
    plt.rcParams['font.family'] = 'cmr10'
    plt.rcParams['font.size'] = 16
    plt.ylabel("Intraclass Distance")
    plt.title("Intraclass Variability" + (" (95% patient-cluster bootstrap CI)" if intraclass_ci is not None else ""))
    plt.tight_layout()
    plt.show()
    sns.boxplot(data=intraclassdistances, palette = [cmap[i] for i in cmap.keys()])
    plt.ylabel("Intraclass Distance")
    plt.title("Intraclass Distances")
    plt.tight_layout()
    plt.show()
    print(f"Interclass distances:\n{interclassdist}\n")

    fdrlabels = [f"{metadata['info']} {classes[i]} - {metadata['info']} {classes[j]}" for i, j in pairidx]
    if fdr_ci is not None:
        fdrdf = pd.DataFrame(
            {
                "FDR_ij": [fdr[i, j] for i, j in pairidx],
                "95% CI low": [fdr_ci[i, j, 0] for i, j in pairidx],
                "95% CI high": [fdr_ci[i, j, 1] for i, j in pairidx],
            },
            index=fdrlabels,
        )
    else:
        fdrdf = pd.DataFrame([fdr[i, j] for i, j in pairidx], index=fdrlabels, columns=["FDR_ij"])
    print(f"Fisher-like discriminability ratio (FDR_ij = d_ij^2 / (v_i + v_j), variance-based):\n{fdrdf}\n")

    if metadata["extended"] == "Yes":
        extresult = extendedseparation(latents, labels, patientids, n_replicates=nboot, randomstate=42)
        extclasses = extresult["classes"]
        extlabels = [f"AMD {c}" for c in extclasses] if metadata["info"] == 'AMD' else list(extclasses)
        extpairidx = [(i, j) for i in range(len(extclasses)) for j in range(i + 1, len(extclasses))]
        extrows = []
        for i, j in extpairidx:
            extrows.append({
                "Pair": f"{extlabels[i]} - {extlabels[j]}",
                "Mahalanobis^2": extresult["mahalanobis"][i, j],
                "Mahalanobis CI low": extresult["mahalanobis_ci"][0][i, j],
                "Mahalanobis CI high": extresult["mahalanobis_ci"][1][i, j],
                "Frechet^2": extresult["frechet"][i, j],
                "Frechet CI low": extresult["frechet_ci"][0][i, j],
                "Frechet CI high": extresult["frechet_ci"][1][i, j],
                "MMD^2": extresult["mmd"][i, j],
                "MMD CI low": extresult["mmd_ci"][0][i, j],
                "MMD CI high": extresult["mmd_ci"][1][i, j],
                "MMD permutation p": extresult["mmd_pvalue"][i, j],
                "MMD permutation p (BH-FDR)": extresult["mmd_pvalue_bh"][i, j],
                "Distinguishable from chance (BH 0.05)": bool(extresult["mmd_pvalue_bh"][i, j] < 0.05),
            })
        extdf = pd.DataFrame(extrows)
        print(f"Extended separation metrics (MMD RBF gamma={extresult['gamma']:.4g}, {nboot} bootstrap replicates):")
        display(extdf)

if metadata["mode"] == 'Correlation':
    totalcorr = []
    totalp = []
    current_labels = []
    for i in range(latents.shape[1]):
        if len(np.array(labels).shape) == 2:
            if i < latents.shape[1] // 2: lbls = [x[0] for x in labels]
            else: lbls = [x[1] for x in labels]
        else: lbls = labels
        corr, pvalue = correlationanalysis(latents[:, i], lbls, method=metadata["stat"])
        totalcorr.append(corr)
        totalp.append(pvalue)
        current_labels.append(lbls)
    reject, p_adjusted = bhfdr(totalp, alpha=0.05)
    correlations = pd.DataFrame(
        {
            "Feature": [
                f"{idx}" for idx in range(latents.shape[1])
            ],
            f"{metadata['stat']} Statistic": totalcorr,
            "p-value": totalp,
            "p-value (BH Adjusted)": p_adjusted,
        }
    )
    print("--- All Features Analysis ---")
    display(correlations)
    significant = correlations[correlations["p-value (BH Adjusted)"] <= 0.05].copy()
    if metadata["stat"] in ("Mann-Whitney", "Brunner-Munzel") and not significant.empty:
        spearman_rhos = []
        for idx in significant.index:
            rho, _ = spearmanr(latents[:, idx], current_labels[idx])
            spearman_rhos.append(rho)
        significant["Spearman Correlation (Rho)"] = spearman_rhos
        significant = significant.sort_values(by="Spearman Correlation (Rho)", ascending=False)
    else:
        significant = significant.rename(columns={f"{metadata['stat']} Statistic": "Spearman Correlation (Rho)"})
        significant = significant.sort_values(by="Spearman Correlation (Rho)", ascending=False)
    print("\nSignificant Features (FDR Corrected)")
    display(significant)

    nboot = int(metadata["boot"]) if metadata["boot"] else 0
    if nboot > 0 and not significant.empty:
        patientids = np.array([str(Path(p).parts[-5]) for p in pathlist])
        rng = np.random.default_rng(42)
        bootrows = []
        for idx in significant.index:
            dimvals = latents[:, idx]
            lblsarr = np.asarray(current_labels[idx])
            reps = np.full(nboot, np.nan)
            for b in range(nboot):
                bidx = patientbootstrapindex(patientids, rng)
                reps[b], _ = correlationanalysis(dimvals[bidx], lblsarr[bidx], method=metadata["stat"])
            low, high, pboot, nok = bootstrapci(reps, alpha=0.05)
            bootrows.append({
                "Feature": correlations.loc[idx, "Feature"],
                "Original effect size": correlations.loc[idx, f"{metadata['stat']} Statistic"],
                "Bootstrap median": float(np.nanmedian(reps)) if nok else np.nan,
                "Bootstrap CI low": low,
                "Bootstrap CI high": high,
                "Bootstrap p-value": pboot,
                "Valid replicates": nok,
            })
        bootdf = pd.DataFrame(bootrows)
        if not bootdf.empty:
            reject_b, p_adj_b = bhfdr(bootdf["Bootstrap p-value"].to_numpy())
            bootdf["Bootstrap p (BH-FDR)"] = p_adj_b
            bootdf["Significant bootstrap (BH)"] = reject_b
            bootdf["Status change"] = np.where(
                ~bootdf["Significant bootstrap (BH)"], "LOST significance", "unchanged"
            )
            print(f"\nPatient-cluster bootstrap ({nboot} replicates) on the {len(bootdf)} BH-FDR-significant dimension(s):")
            display(bootdf)